# چالش ۳: تصمیم پرهزینه

این نوتبوک فقط روایت‌گر است؛ منطق اصلی در پوشه `src/credit_risk` قرار دارد.

## ۰. آماده‌سازی

کتابخانه‌ها، seed ثابت و مسیرها یک‌بار در همین سلول تعریف می‌شوند.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from credit_risk import settings
from credit_risk.settings import SEED, BASE_FEATURES, NEW_FEATURES, TARGET
from credit_risk import db, costs, evaluation, leakage, missingness
from credit_risk.features import build_features
from credit_risk.splitting import split_holdout, HoldoutVault
from credit_risk.baseline import evaluate_baselines, reduction_vs_baseline
from credit_risk.models import build_pipeline
from credit_risk.ablation import score_feature_set, drop_one_table
from credit_risk.importance import top_features
from credit_risk.artifacts import save_artifact, load_artifact
from credit_risk.predict import predict_frame

np.random.seed(SEED)
START_TIME = time.time()

## ۱. خواندن داده از Database

داده فقط از دیتابیس و با SQLAlchemy خوانده می‌شود. ابتدا view پاک‌سازی ساخته می‌شود (کدهای نگهبان و outlierهای غیرممکن با قانون ثابت دامنه‌ای)؛ سپس داده از view خوانده می‌شود.

In [ ]:
engine = db.get_engine()
db.create_clean_view(engine)
loaded_rows = db.count_rows(engine, settings.SCHEMA, settings.TRAIN_TABLE)
quarantine_rows = db.count_rows(engine, settings.SCHEMA, settings.TRAIN_TABLE + "_quarantine")
print("rows in table:", loaded_rows, "| rows in quarantine:", quarantine_rows)

In [ ]:
frame = db.load_clean_frame(engine)
frame = build_features(frame)
print(frame.shape)
frame.head()

## ۲. جداسازی مجموعه نهایی

پیش از هر بررسی، بخشی از داده کنار گذاشته و در `HoldoutVault` قفل می‌شود. این مجموعه فقط یک‌بار در بخش آخر باز می‌شود و هر باز شدن در فایل log ثبت می‌شود.

In [ ]:
dev, holdout = split_holdout(frame)
leakage.assert_disjoint_ids(dev[settings.ID_COLUMN], holdout[settings.ID_COLUMN])
vault = HoldoutVault(holdout, settings.CURATED_DIR / "holdout_access.log")
print("dev rows:", len(dev), "| holdout rows:", len(holdout))
print("default rate dev:", round(dev[TARGET].mean(), 4), "| holdout:", round(holdout[TARGET].mean(), 4))
del holdout

## ۳. شناخت داده (فقط روی dev)

ساختار ستون‌ها، نسبت کلاس‌ها، تعداد ردیف‌های دارای کد نگهبان و تعداد مقادیر سقف‌خورده.

In [ ]:
print(dev.dtypes)
print("class ratio:\n", dev[TARGET].value_counts(normalize=True))
print("rows with sentinel code:", int(dev["has_delinquency_code"].sum()))
print("rows with capped values:", int(dev["util_was_capped"].sum()))
dev[BASE_FEATURES].describe().T

## ۴. الگوی گم‌شدگی

آیا نبودن مقدار خودش اطلاعات دارد؟ نرخ نکول و تفاوت استانداردشده سایر ستون‌ها بین ردیف‌های دارای و فاقد مقدار بررسی می‌شود.

**نتیجه‌گیری من:** (اینجا با شاهد عددی خودتان بنویسید)

In [ ]:
missing_columns = ["monthly_income", "dependents"]
print(missingness.missing_summary(dev, missing_columns))
for column in missing_columns:
    print(missingness.default_rate_by_missing(dev, column))
other_columns = ["age", "debt_ratio", "revolving_utilization", "open_credit_lines", "past_due_90"]
print(missingness.standardized_gap(dev, "monthly_income", other_columns))

## ۵. مقایسه سیاست‌های گم‌شدگی

سه سیاست: میانه، میانه به‌همراه indicator، و مدیریت بومی توسط مدل boosting. مقایسه با هزینه و AUC روی داده اعتبارسنجی چندتکه‌ای.

In [ ]:
all_columns = BASE_FEATURES + NEW_FEATURES
policy_runs = [
    ("logistic", "median"),
    ("logistic", "median_indicator"),
    ("boosting", "median"),
    ("boosting", "native"),
]
policy_rows = []
for model_name, policy in policy_runs:
    row = score_feature_set(model_name, dev, all_columns, policy)
    policy_rows.append({"model": model_name, "policy": policy, **row})
policy_table = pd.DataFrame(policy_rows)
policy_table

## ۶. خط مبنا

قاعده‌های ساده بدون یادگیری. بهترین قاعده، سطح زمین است.

In [ ]:
baseline_table = evaluate_baselines(dev)
baseline_cost = baseline_table.loc[0, "cost_per_1000"]
print(baseline_table)

## ۷. دو خانواده مدل

رگرسیون لجستیک (حساس به مقیاس) و boosting درختی (بدون نیاز به مقیاس). برای هرکدام سیاست گم‌شدگی را از جدول بخش ۵ انتخاب می کنیم..

In [ ]:
POLICY = {"logistic": "median_indicator", "boosting": "native"}
y_dev = dev[TARGET]
oof_results = {}
model_rows = []
for model_name in ("logistic", "boosting"):
    pipeline = build_pipeline(model_name, all_columns, policy=POLICY[model_name])
    result = evaluation.out_of_fold_proba(pipeline, dev[all_columns], y_dev)
    threshold, cost = costs.best_threshold(y_dev.to_numpy(), result.proba)
    oof_results[model_name] = result
    model_rows.append({"model": model_name, "best_threshold": threshold,
                       "cost_per_1000": cost,
                       "reduction_vs_baseline": reduction_vs_baseline(cost, baseline_cost)})
model_table = pd.DataFrame(model_rows)
model_table

In [ ]:
BEST_MODEL = model_table.sort_values("cost_per_1000").iloc[0]["model"]
best_result = oof_results[BEST_MODEL]
print("selected model:", BEST_MODEL)

## ۸. انتخاب آستانه از روی هزینه

منحنی هزینه نسبت به آستانه، آستانه بهینه، آستانه نظری `1/13` و حساسیت به تغییر ماتریس هزینه.

In [ ]:
y_values = y_dev.to_numpy()
curve = costs.cost_curve(y_values, best_result.proba, costs.threshold_grid())
THRESHOLD, VALIDATION_COST = costs.best_threshold(y_values, best_result.proba)
print("chosen threshold:", THRESHOLD, "| cost per 1000:", round(VALIDATION_COST, 1))
print("theoretical threshold:", round(costs.theoretical_threshold(), 4))
plt.plot(curve["threshold"], curve["cost_per_1000"])
plt.axvline(THRESHOLD, color="red")
plt.xlabel("threshold")
plt.ylabel("cost per 1000 applicants")
plt.show()

In [ ]:
sensitivity = costs.threshold_sensitivity(y_values, best_result.proba, [12, 8, 4, 2])
sensitivity

## ۹. اثر هر ویژگی ساخته‌شده

هر ویژگی جدید حذف و تغییر هزینه و AUC سنجیده می‌شود. ویژگی بی‌اثر باید حذف و گزارش شود.

In [ ]:
ablation_table = drop_one_table(BEST_MODEL, dev, all_columns, NEW_FEATURES, POLICY[BEST_MODEL])
ablation_table

In [ ]:
# Fill in after reading the table above
DROPPED_FEATURES = []
FINAL_COLUMNS = [name for name in all_columns if name not in DROPPED_FEATURES]
FINAL_NEW_FEATURES = [name for name in NEW_FEATURES if name not in DROPPED_FEATURES]

## ۱۰. ممیزی نشت داده

جدول زمان در دسترس بودن هر ویژگی، و بازرسی این‌که آماره‌های آماده‌سازی فقط از بخش آموزش آمده‌اند.

In [ ]:
leakage.audit_table()

In [ ]:
train_part = dev.sample(frac=0.8, random_state=SEED)
valid_part = dev.drop(train_part.index)
leakage.assert_disjoint_ids(train_part[settings.ID_COLUMN], valid_part[settings.ID_COLUMN])
if POLICY[BEST_MODEL] != "native":
    check_pipeline = build_pipeline(BEST_MODEL, FINAL_COLUMNS, policy=POLICY[BEST_MODEL])
    check_pipeline.fit(train_part[FINAL_COLUMNS], train_part[TARGET])
    leakage.assert_fit_uses_train_only(check_pipeline, train_part, FINAL_COLUMNS)
    print("imputer statistics come from the training part only")

## ۱۱. پایداری چندتکه‌ای

معیارها در هر fold با آستانه ثابت، و پراکندگی آن‌ها. همچنین Accuracy برای نشان دادن گمراه‌کننده بودن آن.

In [ ]:
final_pipeline = build_pipeline(BEST_MODEL, FINAL_COLUMNS, policy=POLICY[BEST_MODEL])
final_oof = evaluation.out_of_fold_proba(final_pipeline, dev[FINAL_COLUMNS], y_dev)
THRESHOLD, VALIDATION_COST = costs.best_threshold(y_values, final_oof.proba)
fold_table = evaluation.per_fold_table(y_values, final_oof, THRESHOLD)
print(fold_table.round(4))
evaluation.stability_summary(fold_table)

In [ ]:
print("accuracy of approving everybody:", round(evaluation.accuracy_of_constant_healthy(y_values), 4))
print("accuracy of the model:", round(fold_table["accuracy"].mean(), 4))

## ۱۲. مهم‌ترین ویژگی‌ها

اهمیت جایگشتی روی بخش اعتبارسنجی.

In [ ]:
fitted = build_pipeline(BEST_MODEL, FINAL_COLUMNS, policy=POLICY[BEST_MODEL])
fitted.fit(train_part[FINAL_COLUMNS], train_part[TARGET])
importance_table = top_features(fitted, valid_part[FINAL_COLUMNS], valid_part[TARGET])
importance_table.head(10)

## ۱۳. انصاف بین گروه‌های سنی

معیارها در هر گروه سنی با همان آستانه.

In [ ]:
dev["age_band"] = pd.cut(dev["age"], [17, 30, 45, 60, 120])
fairness_table = evaluation.subgroup_table(dev, final_oof.proba, THRESHOLD, "age_band")
fairness_table.round(4)

## ۱۴. مدل نهایی، ذخیره و باز کردن مجموعه نهایی (یک‌بار)

مدل روی کل dev آموزش می‌بیند و ذخیره می‌شود. سپس مجموعه نهایی فقط یک‌بار باز می‌شود.

In [ ]:
final_model = build_pipeline(BEST_MODEL, FINAL_COLUMNS, policy=POLICY[BEST_MODEL])
final_model.fit(dev[FINAL_COLUMNS], y_dev)
model_path = save_artifact(final_model, THRESHOLD, FINAL_COLUMNS, FINAL_NEW_FEATURES)
print("saved:", model_path)

In [ ]:
holdout_frame = vault.open("final evaluation of the selected model")
holdout_proba = final_model.predict_proba(holdout_frame[FINAL_COLUMNS])[:, 1]
holdout_metrics = evaluation.classification_metrics(
    holdout_frame[TARGET].to_numpy(), holdout_proba, THRESHOLD)
holdout_baseline_cost = evaluate_baselines(holdout_frame).loc[0, "cost_per_1000"]
print(pd.Series(holdout_metrics).round(4))
print("validation cost per 1000:", round(VALIDATION_COST, 1))
print("holdout cost per 1000   :", round(holdout_metrics["cost_per_1000"], 1))
print("baseline cost on holdout:", round(holdout_baseline_cost, 1))
print("reduction vs baseline   :", round(reduction_vs_baseline(
    holdout_metrics["cost_per_1000"], holdout_baseline_cost), 3))

## ۱۵. استفاده از مدل ذخیره‌شده روی ورودی تازه

نمونه‌ای از مسیر پیش‌بینی با artifact ذخیره‌شده.

In [ ]:
artifact = load_artifact()
sample_predictions = predict_frame(holdout_frame.head(5), artifact)
sample_predictions

In [ ]:
print("notebook runtime (minutes):", round((time.time() - START_TIME) / 60, 2))